# Sentinelle V1 — XGBoost sur GPU (répétition Kaggle)
Équipe ImpactBoyz, GPU Challenge 2026. PaySim sert de trafic de fond, nos scénarios d'arnaque y sont injectés, puis XGBoost note le compte destinataire de chaque transfert. Réglages : accélérateur **GPU T4 x2**, jeu de données **ealaxi/paysim1**.

In [ ]:
# === 1. Configuration et chargement de PaySim ================================
import glob, json, time
import numpy as np, pandas as pd

SEED = 42
rng = np.random.default_rng(SEED)
N_PAR_SCENARIO = 300            # instances injectées par scénario
SCENARIO_CACHE = "S4_chaine"    # jamais montré à l'entraînement
JOUR = 24 * 3600                # tout le temps est compté en secondes

chemins = glob.glob("/kaggle/input/**/PS_*_log.csv", recursive=True)
assert chemins, "Ajoute le jeu 'ealaxi/paysim1' au notebook (Add Input)."

# Les colonnes de solde ne sont JAMAIS chargées : l'auteur de PaySim écrit qu'elles
# ne doivent pas être utilisées (fuite de données).
t0 = time.perf_counter()
ps = pd.read_csv(chemins[0], usecols=["step", "type", "amount", "nameOrig", "nameDest", "isFraud"])
print(f"PaySim : {len(ps):,} lignes lues en {time.perf_counter() - t0:.1f} s")

# Les fraudes natives de PaySim ne sont pas des comptes mules (mesuré dans le guide) :
# on les retire pour que l'étiquette ne désigne que nos scénarios.
ps = ps[ps["isFraud"] == 0]
fond = pd.DataFrame({
    # PaySim avance par pas d'une heure : on tire la seconde au hasard dans l'heure.
    "t": (ps["step"].to_numpy() - 1) * 3600 + rng.integers(0, 3600, len(ps)),
    "type": ps["type"].to_numpy(),
    "montant": ps["amount"].to_numpy(),
    "src": ps["nameOrig"].to_numpy(),
    "dst": ps["nameDest"].to_numpy(),
    "y": 0,
    "scen": "fond",
})
T_MAX = int(fond["t"].max())
tr = fond[fond["type"] == "TRANSFER"]
CLIENTS = tr["dst"].unique()                                   # comptes clients existants
Q_MONTANT = np.quantile(tr["montant"], np.linspace(0, 1, 1001))  # montants réalistes
del ps
print(f"Trafic de fond : {len(fond):,} lignes, dont {len(tr):,} transferts, sur {T_MAX / JOUR:.0f} jours")

In [ ]:
# === 2. Injection des scénarios d'arnaque et des contre-exemples =============
# Déroulés : docs/scenarios-fraude.md. Délais, nombre de mules et montants sont des
# HYPOTHÈSES DE SIMULATION tirées au hasard, à annoncer comme telles au jury.
L = []
_n = iter(range(10**9))
AGENTS = [f"A{i:05d}" for i in range(400)]

def nouveau():  return f"X{next(_n):09d}"                 # compte créé pour le scénario
def client():   return CLIENTS[rng.integers(len(CLIENTS))]  # compte déjà présent dans PaySim
def agent():    return AGENTS[rng.integers(len(AGENTS))]
def expediteur():                                         # comme dans PaySim : presque jamais d'historique
    return nouveau() if rng.random() < 0.9 else client()
def mn(a, b):   return int(rng.uniform(a, b) * 60)        # durée en secondes, bornes en minutes
def montant(lo=0.10, hi=0.98):                            # montant pris dans la distribution PaySim
    return float(Q_MONTANT[rng.integers(int(lo * 1000), int(hi * 1000))])
def debut(marge=4):
    return int(rng.uniform(JOUR, T_MAX - marge * JOUR))
def tx(t, typ, a, src, dst, y, scen):
    if 0 <= t <= T_MAX:
        L.append((int(t), typ, round(float(a), 2), src, dst, y, scen))

def habiller(compte, t, p=0.5):
    """Petit historique normal avant l'arnaque, pour qu'une mule ne soit pas toujours un compte vierge."""
    if rng.random() < p:
        for _ in range(rng.integers(1, 5)):
            tx(t - rng.uniform(2, 20) * JOUR, "TRANSFER", montant(0.05, 0.5), expediteur(), compte, 0, "fond_mule")

def s1_erreur(n):       # la victime « rembourse » elle-même un faux dépôt
    equipes = []
    for _ in range(n):
        if equipes and rng.random() < 0.5:        # une équipe de mules ressert pour une autre victime
            m1, m2, t = equipes[rng.integers(len(equipes))]
            t += int(rng.uniform(0.2, 4) * JOUR)
        else:
            m1, m2, t = nouveau(), nouveau(), debut()
            habiller(m1, t)
        equipes.append((m1, m2, t))
        a, part = montant(), rng.uniform(0.4, 0.7)
        tx(t, "TRANSFER", a, expediteur(), m1, 1, "S1_erreur")
        t2 = t + mn(2, 15)
        tx(t2, "TRANSFER", a * part, m1, m2, 1, "S1_erreur")
        tx(t2 + mn(1, 10), "CASH_OUT", a * (1 - part) * 0.98, m1, agent(), 1, "S1_erreur")
        tx(t2 + mn(3, 20), "CASH_OUT", a * part * 0.98, m2, agent(), 1, "S1_erreur")

def s2_faux_agent(n):   # le fraudeur vide le compte en rafale vers 1 ou 2 mules
    for _ in range(n):
        t, v = debut(), expediteur()
        mules = [nouveau() for _ in range(rng.integers(1, 3))]
        for m in mules: habiller(m, t)
        recu = dict.fromkeys(mules, 0.0)
        for _ in range(rng.integers(2, 6)):
            m, a = mules[rng.integers(len(mules))], montant(0.2, 0.8)
            tx(t, "TRANSFER", a, v, m, 1, "S2_faux_agent")
            recu[m] += a
            t += mn(0.5, 3)
        for m, a in recu.items():
            if a: tx(t + mn(3, 20), "CASH_OUT", a * 0.98, m, agent(), 1, "S2_faux_agent")

def s3_collecteur(n):   # une même mule reçoit plusieurs victimes en 24 à 72 h
    for _ in range(n):
        t0, m = debut(), nouveau()
        habiller(m, t0)
        moments = np.sort(t0 + rng.uniform(0, rng.uniform(1, 3) * JOUR, rng.integers(4, 13)))
        pot = 0.0
        for i, t in enumerate(moments):
            a = montant(0.1, 0.9)
            tx(t, "TRANSFER", a, expediteur(), m, 1, "S3_collecteur")
            pot += a
            if rng.random() < 0.6 or i == len(moments) - 1:   # la mule ne garde presque rien
                if rng.random() < 0.25:
                    m2, t2 = nouveau(), t + mn(2, 15)
                    tx(t2, "TRANSFER", pot, m, m2, 1, "S3_collecteur")
                    tx(t2 + mn(3, 20), "CASH_OUT", pot * 0.98, m2, agent(), 1, "S3_collecteur")
                else:
                    tx(t + mn(2, 30), "CASH_OUT", pot * 0.98, m, agent(), 1, "S3_collecteur")
                pot = 0.0

def s4_chaine(n):       # l'argent traverse 2 à 4 étages avant le retrait
    for _ in range(n):
        t, m = debut(), nouveau()
        habiller(m, t, p=1.0)                     # la première mule a toujours un passé normal
        a = montant(0.3, 0.98)
        tx(t, "TRANSFER", a, expediteur(), m, 1, "S4_chaine")
        etage = [(m, a, t)]
        for _ in range(rng.integers(1, 4)):
            suivant = []
            for src, a, t in etage:
                parts = [rng.uniform(0.4, 0.6)] if rng.random() < 0.5 else [1.0]
                parts += [1 - parts[0]] if parts[0] < 1 else []
                for p in parts:
                    m2, t2 = nouveau(), t + mn(1, 8)
                    tx(t2, "TRANSFER", a * p * 0.99, src, m2, 1, "S4_chaine")
                    suivant.append((m2, a * p * 0.99, t2))
            etage = suivant
        for src, a, t in etage:
            tx(t + mn(2, 15), "CASH_OUT", a * 0.98, src, agent(), 1, "S4_chaine")

# --- Contre-exemples légitimes (y = 0) : ils ressemblent à une mule sans en être une.
def l_tontine(n):       # mêmes expéditeurs chaque semaine, pot reversé à un membre
    for _ in range(n):
        chef = client()
        membres = [expediteur() for _ in range(rng.integers(8, 16))]
        a, t0 = montant(0.02, 0.3), int(rng.uniform(0, 6 * JOUR))
        for semaine in range(4):
            t = t0 + semaine * 7 * JOUR
            for mb in membres:
                tx(t + rng.uniform(0, 6 * 3600), "TRANSFER", a, mb, chef, 0, "L_tontine")
            tx(t + rng.uniform(8, 30) * 3600, "TRANSFER", a * len(membres), chef, membres[semaine % len(membres)], 0, "L_tontine")

def l_commerce(n):      # beaucoup de clients nouveaux, petits montants, solde qui reste
    for _ in range(n):
        boutique = client() if rng.random() < 0.5 else nouveau()
        caisse = 0.0
        for jour in np.sort(rng.integers(0, 30, rng.integers(20, 61))):
            a = montant(0.01, 0.25)
            tx(jour * JOUR + rng.uniform(8, 20) * 3600, "TRANSFER", a, expediteur(), boutique, 0, "L_commerce")
            caisse += a
            if rng.random() < 0.1:
                tx((jour + 1) * JOUR + rng.uniform(8, 12) * 3600, "CASH_OUT", caisse * rng.uniform(0.3, 0.7), boutique, agent(), 0, "L_commerce")
                caisse = 0.0

def l_etudiant(n):      # compte récent, retrait rapide, mais toujours la même famille
    for _ in range(n):
        etu, famille, t = nouveau(), [expediteur() for _ in range(rng.integers(1, 3))], debut(15)
        for _ in range(rng.integers(2, 5)):
            a = montant(0.05, 0.6)
            tx(t, "TRANSFER", a, famille[rng.integers(len(famille))], etu, 0, "L_etudiant")
            tx(t + mn(10, 90), "CASH_OUT", a * rng.uniform(0.7, 1.0), etu, agent(), 0, "L_etudiant")
            t += int(rng.uniform(3, 10) * JOUR)

def l_nouveau(n):       # compte neuf ordinaire : reçoit 1 à 3 transferts, retire parfois, rarement vite
    for _ in range(n):
        cpt, t = nouveau(), debut(1)
        for _ in range(rng.integers(1, 4)):
            a = montant(0.05, 0.98)
            tx(t, "TRANSFER", a, expediteur(), cpt, 0, "L_nouveau")
            if rng.random() < 0.35:
                attente = mn(5, 30) if rng.random() < 0.3 else mn(120, 3000)
                tx(t + attente, "CASH_OUT", a * rng.uniform(0.5, 1.0), cpt, agent(), 0, "L_nouveau")
            t += int(rng.uniform(0.1, 6) * JOUR)

for f in (s1_erreur, s2_faux_agent, s3_collecteur, s4_chaine):
    f(N_PAR_SCENARIO)
l_tontine(N_PAR_SCENARIO); l_commerce(N_PAR_SCENARIO); l_etudiant(2 * N_PAR_SCENARIO); l_nouveau(10 * N_PAR_SCENARIO)

inj = pd.DataFrame(L, columns=fond.columns)
tout = pd.concat([fond, inj], ignore_index=True).sort_values("t", kind="stable", ignore_index=True)
print(inj.groupby("scen").agg(lignes=("y", "size"), transferts=("type", lambda s: (s == "TRANSFER").sum())))
tf = tout[tout["type"] == "TRANSFER"]
print(f"\nTransferts à noter : {len(tf):,}, dont {tf['y'].sum():,} frauduleux ({tf['y'].mean():.2%})")

In [ ]:
# === 3. Signaux du compte destinataire : passé uniquement, sans soldes ========
# Le même code tourne sur CPU (numpy) et sur GPU (cupy) : c'est notre première mesure du gain.
K = 1 << 23                     # > durée totale en secondes (31 jours = 2 678 400)

class Histo:
    """Événements d'un compte triés par (compte, t). Répond : combien, quelle somme, AVANT t."""
    def __init__(self, xp, compte, t, valeur=None):
        cle = compte * K + t
        o = xp.argsort(cle)
        self.xp, self.cle = xp, cle[o]
        v = xp.ones(len(cle)) if valeur is None else valeur[o].astype(xp.float64)
        self.cs = xp.concatenate([xp.zeros(1), xp.cumsum(v)])
    def bornes(self, compte, t, fenetre=None):
        xp = self.xp
        depuis = 0 if fenetre is None else xp.maximum(t - fenetre, 0)
        lo = xp.searchsorted(self.cle, compte * K + depuis, side="left")
        hi = xp.searchsorted(self.cle, compte * K + t, side="left")     # strictement avant t
        return lo, hi
    def nb(self, compte, t, fenetre=None):
        lo, hi = self.bornes(compte, t, fenetre); return (hi - lo).astype(self.xp.float32)
    def somme(self, compte, t, fenetre=None):
        lo, hi = self.bornes(compte, t, fenetre); return self.cs[hi] - self.cs[lo]
    def depuis_dernier(self, compte, t):          # -1 si aucun événement passé
        lo, hi = self.bornes(compte, t)
        dernier = self.cle[self.xp.maximum(hi - 1, 0)] % K
        return self.xp.where(hi > lo, t - dernier, -1).astype(self.xp.float32)
    def depuis_premier(self, compte, t):          # ancienneté ; -1 si compte jamais vu
        lo, hi = self.bornes(compte, t)
        premier = self.cle[self.xp.minimum(lo, len(self.cle) - 1)] % K
        return self.xp.where(hi > lo, t - premier, -1).astype(self.xp.float32)

def rang_dans_paire(xp, s, d, t):
    """Nombre de transferts déjà faits de s vers d avant celui-ci."""
    paire = s * (1 << 24) + d
    o = np.lexsort((t, paire)) if xp is np else xp.lexsort(xp.stack([t, paire]))
    p = paire[o]
    rang = xp.empty(len(p), dtype=xp.int64)
    rang[o] = xp.arange(len(p)) - xp.searchsorted(p, p, side="left")
    return rang

def signaux(xp, t, src, dst, montant, est_tr, est_sortie):
    """Une ligne de signaux par TRANSFERT (le moment où Sentinelle note le destinataire)."""
    t, src, dst, a = (xp.asarray(v) for v in (t, src, dst, montant))
    tr, so = xp.asarray(est_tr), xp.asarray(est_sortie)
    T, S, D, A = t[tr], src[tr], dst[tr], a[tr]
    premier_contact = (rang_dans_paire(xp, S, D, T) == 0)
    vu      = Histo(xp, xp.concatenate([src, dst]), xp.concatenate([t, t]))   # toute activité
    entree  = Histo(xp, D, T, A)                                # transferts reçus
    neuf    = Histo(xp, D, T, premier_contact)                  # ... venant d'un expéditeur jamais vu
    sortie  = Histo(xp, src[so], t[so], a[so])                  # transferts envoyés et retraits
    rapide  = Histo(xp, src[so], t[so], entree.nb(src[so], t[so], 1800) > 0)  # sortie < 30 min après une entrée
    envoi   = Histo(xp, S, T, A)                                # transferts envoyés seulement
    nb_in, nb_out, som_in = entree.nb(D, T), sortie.nb(D, T), entree.somme(D, T)
    f = {
        "montant": A,
        "premier_contact": premier_contact,
        "dst_anciennete_j": xp.where(vu.depuis_premier(D, T) < 0, -1, vu.depuis_premier(D, T) / JOUR),
        "dst_nb_recus": nb_in,
        "dst_nb_recus_24h": entree.nb(D, T, JOUR),
        "dst_somme_recue_24h": entree.somme(D, T, JOUR),
        "dst_nouveaux_exp_24h": neuf.somme(D, T, JOUR),
        "dst_nouveaux_exp_72h": neuf.somme(D, T, 3 * JOUR),
        "dst_part_nouveaux_exp": neuf.somme(D, T) / xp.maximum(nb_in, 1),
        "dst_nb_sorties": nb_out,
        "dst_ratio_sorties_entrees": sortie.somme(D, T) / (som_in + 1),
        "dst_sorties_rapides": rapide.somme(D, T),
        "dst_part_sorties_rapides": rapide.somme(D, T) / xp.maximum(nb_out, 1),
        "dst_min_depuis_entree": entree.depuis_dernier(D, T) / 60,
        "montant_vs_habitude_dst": xp.where(nb_in > 0, A * nb_in / (som_in + 1), -1),
        "src_anciennete_j": xp.where(vu.depuis_premier(S, T) < 0, -1, vu.depuis_premier(S, T) / JOUR),
        "src_nb_envois": envoi.nb(S, T),
        "src_nb_envois_10min": envoi.nb(S, T, 600),
        "src_recu_30min_vs_montant": entree.somme(S, T, 1800) / (A + 1),
    }
    vers_np = (lambda v: v) if xp is np else xp.asnumpy
    return pd.DataFrame({k: vers_np(v).astype(np.float32) for k, v in f.items()})

codes, comptes = pd.factorize(pd.concat([tout["src"], tout["dst"]], ignore_index=True))
assert len(comptes) < (1 << 24) and T_MAX < K
n = len(tout)
EV = dict(t=tout["t"].to_numpy(np.int64), src=codes[:n].astype(np.int64), dst=codes[n:].astype(np.int64),
          montant=tout["montant"].to_numpy(np.float64),
          est_tr=(tout["type"] == "TRANSFER").to_numpy(),
          est_sortie=tout["type"].isin(["TRANSFER", "CASH_OUT"]).to_numpy())

temps = {}
t0 = time.perf_counter(); X = signaux(np, **EV); temps["signaux_cpu"] = time.perf_counter() - t0
print(f"Signaux sur CPU (numpy) : {temps['signaux_cpu']:.1f} s pour {len(X):,} transferts, {X.shape[1]} signaux")
try:
    import cupy as cp
    signaux(cp, **{k: v[:1000] for k, v in EV.items()})            # échauffement du GPU
    t0 = time.perf_counter(); Xg = signaux(cp, **EV); temps["signaux_gpu"] = time.perf_counter() - t0
    print(f"Signaux sur GPU (cupy)  : {temps['signaux_gpu']:.1f} s, soit x{temps['signaux_cpu'] / temps['signaux_gpu']:.1f}")
    assert np.allclose(X, Xg, rtol=1e-3, atol=1e-3), "CPU et GPU ne donnent pas les mêmes signaux"
except Exception as e:
    print("GPU indisponible pour les signaux :", repr(e)[:200])

# Test anti-fuite : en coupant les données au jour 15, les signaux d'avant le jour 15 ne bougent pas.
coupe = EV["t"] < 15 * JOUR
Xc = signaux(np, **{k: v[coupe] for k, v in EV.items()})
assert np.allclose(X[EV["t"][EV["est_tr"]] < 15 * JOUR].to_numpy(), Xc.to_numpy(), rtol=1e-4, atol=1e-2), "Un signal lit le futur !"
print("Test anti-fuite réussi : aucun signal ne lit le futur, aucune colonne de solde n'est chargée.")

In [ ]:
# === 4. XGBoost : découpage dans le temps, CPU puis GPU =======================
import xgboost as xgb
from sklearn.metrics import average_precision_score

meta = tout.loc[EV["est_tr"], ["t", "y", "scen"]].reset_index(drop=True)
jour = meta["t"] / JOUR
app = (jour < 20) & (meta["scen"] != SCENARIO_CACHE)      # entraînement : jours 0 à 19, scénario caché retiré
val = (jour >= 20) & (jour < 24) & (meta["scen"] != SCENARIO_CACHE)
test = jour >= 24                                         # test : jours 24 à 30, scénario caché compris
y = meta["y"].to_numpy()
print(f"Entraînement {app.sum():,} ({y[app].sum():,} fraudes) | validation {val.sum():,} ({y[val].sum():,}) | test {test.sum():,} ({y[test].sum():,})")

def entrainer(device, Xa, ya, arbres=512, arret=50, **eval_):
    m = xgb.XGBClassifier(n_estimators=arbres, max_depth=6, learning_rate=0.1, tree_method="hist",
                          device=device, eval_metric="aucpr", early_stopping_rounds=arret,
                          scale_pos_weight=float((ya == 0).sum() / max(ya.sum(), 1)), random_state=SEED)
    t0 = time.perf_counter()
    m.fit(Xa, ya, verbose=False, **eval_)
    return m, time.perf_counter() - t0

# Sans GPU, XGBoost bascule sur le CPU avec un simple avertissement : on vérifie donc nous-mêmes.
import subprocess
GPU = subprocess.run("nvidia-smi", shell=True, capture_output=True).returncode == 0
if GPU:
    entrainer("cuda", X[app][:1000], y[app][:1000], arbres=5, arret=None)      # échauffement du GPU
else:
    print("Pas de GPU : choisir Settings > Accelerator > GPU T4 x2. Les mesures GPU sont sautées.")

modele, temps["xgb_cpu"] = entrainer("cpu", X[app], y[app], eval_set=[(X[val], y[val])])
print(f"XGBoost CPU : {temps['xgb_cpu']:.1f} s, {modele.best_iteration + 1} arbres retenus")
if GPU:
    modele_gpu, temps["xgb_gpu"] = entrainer("cuda", X[app], y[app], eval_set=[(X[val], y[val])])
    print(f"XGBoost GPU : {temps['xgb_gpu']:.1f} s, {modele_gpu.best_iteration + 1} arbres retenus")

# Mesures sur le test. Seuil d'alerte réglé sur la VALIDATION : 1 % de fausses alertes.
score = modele.predict_proba(X)[:, 1]
SEUIL = float(np.quantile(score[val & (meta["y"] == 0)], 0.99))
res = meta[test].assign(score=score[test], alerte=score[test] >= SEUIL)
print(f"\nPrécision moyenne sur le test : {average_precision_score(res['y'], res['score']):.3f}  (un tirage au hasard donnerait {res['y'].mean():.3f})")
print(f"Seuil d'alerte : {SEUIL:.3f}. Part des transferts alertés, par scénario, sur le test :")
bilan = res.groupby("scen").agg(transferts=("y", "size"), fraudes=("y", "sum"), part_alertee=("alerte", "mean")).round(3)
print(bilan)
print(f"\nScénario caché ({SCENARIO_CACHE}), jamais vu à l'entraînement : {res.loc[res['scen'] == SCENARIO_CACHE, 'alerte'].mean():.1%} de ses transferts sont alertés.")
vierge = res[(res["y"] == 1) & (X.loc[res.index, "dst_nb_recus"] == 0) & (X.loc[res.index, "src_recu_30min_vs_montant"] == 0)]
print(f"Premier transfert d'une victime vers une mule qui n'a encore rien reçu : {vierge['alerte'].mean():.1%} alertés sur {len(vierge)} (cas le plus dur, peu de passé à lire).")
if average_precision_score(res["y"], res["score"]) > 0.99:
    print("ATTENTION : score > 0,99. Soupçonner une fuite ou des scénarios trop faciles avant de le montrer.")

In [ ]:
# === 5. Le gain du GPU sur un volume grossi ===================================
# Sur 30 jours de PaySim, XGBoost s'entraîne en quelques secondes : le GPU paraît inutile.
# On recopie donc le tableau d'entraînement FACTEUR fois, avec un léger bruit, pour CHRONOMÉTRER.
# Ce tableau grossi ne sert qu'à mesurer des temps, jamais à évaluer le modèle.
FACTEUR = 20
Xa, ya = X[app].to_numpy(), y[app]
Xgros = np.tile(Xa, (FACTEUR, 1)) * rng.uniform(0.98, 1.02, (len(Xa) * FACTEUR, 1)).astype(np.float32)
ygros = np.tile(ya, FACTEUR)
print(f"Volume grossi : {len(Xgros):,} lignes x {Xgros.shape[1]} signaux ({Xgros.nbytes / 1e9:.1f} Go)")
_, temps["xgb_gros_cpu"] = entrainer("cpu", Xgros, ygros, arbres=200, arret=None)
print(f"XGBoost CPU, 200 arbres : {temps['xgb_gros_cpu']:.1f} s")
if GPU:
    _, temps["xgb_gros_gpu"] = entrainer("cuda", Xgros, ygros, arbres=200, arret=None)
    print(f"XGBoost GPU, 200 arbres : {temps['xgb_gros_gpu']:.1f} s, soit x{temps['xgb_gros_cpu'] / temps['xgb_gros_gpu']:.1f}")
del Xgros, ygros
print("\nTableau des temps (secondes) :"); print(pd.Series(temps).round(2).to_string())

In [ ]:
# === 6. La raison lisible, et la sauvegarde ==================================
RAISONS = {
    "premier_contact": "l'expéditeur n'a jamais envoyé d'argent à ce compte",
    "dst_anciennete_j": "le compte destinataire est très récent",
    "dst_nb_recus": "le nombre de transferts déjà reçus par ce compte est inhabituel",
    "dst_nb_recus_24h": "ce compte a reçu beaucoup de transferts en 24 heures",
    "dst_somme_recue_24h": "ce compte a reçu beaucoup d'argent en 24 heures",
    "dst_nouveaux_exp_24h": "ce compte reçoit de l'argent de plusieurs inconnus depuis 24 heures",
    "dst_nouveaux_exp_72h": "ce compte reçoit de l'argent de plusieurs inconnus depuis 3 jours",
    "dst_part_nouveaux_exp": "presque tous ceux qui paient ce compte ne l'ont fait qu'une fois",
    "dst_nb_sorties": "ce compte fait sortir de l'argent très souvent",
    "dst_ratio_sorties_entrees": "ce compte ne garde presque rien de ce qu'il reçoit",
    "dst_sorties_rapides": "ce compte a déjà vidé des fonds quelques minutes après les avoir reçus",
    "dst_part_sorties_rapides": "ce compte vide presque toujours les fonds dès qu'il les reçoit",
    "dst_min_depuis_entree": "ce compte vient de recevoir un autre transfert",
    "montant_vs_habitude_dst": "le montant est inhabituel pour ce compte",
    "montant": "le montant est élevé",
    "src_anciennete_j": "le compte expéditeur est très récent",
    "src_nb_envois": "l'expéditeur envoie de l'argent très souvent",
    "src_nb_envois_10min": "l'expéditeur enchaîne plusieurs transferts en quelques minutes",
    "src_recu_30min_vs_montant": "l'expéditeur vient lui-même de recevoir cet argent",
}
assert set(RAISONS) == set(X.columns)

def expliquer(lignes, k=2):
    """Pour chaque transfert : score, décision et les k signaux qui poussent le plus vers la fraude."""
    c = modele.get_booster().predict(xgb.DMatrix(lignes), pred_contribs=True)[:, :-1]
    s = modele.predict_proba(lignes)[:, 1]
    for i in range(len(lignes)):
        tops = [X.columns[j] for j in np.argsort(-c[i])[:k] if c[i, j] > 0]
        decision = "ALERTE" if s[i] >= SEUIL else "laisser passer"
        print(f"score {s[i]:.2f} -> {decision:14s} | " + " ; ".join(RAISONS[n] for n in tops))

for scen in ["S1_erreur", "S3_collecteur", SCENARIO_CACHE, "L_tontine", "fond"]:
    idx = res.index[res["scen"] == scen][:3]
    print(f"\n{scen}"); expliquer(X.loc[idx])

modele.get_booster().save_model("/kaggle/working/sentinelle_v1.ubj")
json.dump({"signaux": list(X.columns), "seuil_alerte": SEUIL, "raisons": RAISONS, "temps_s": temps,
           "scenario_cache": SCENARIO_CACHE, "seed": SEED},
          open("/kaggle/working/sentinelle_v1.json", "w"), ensure_ascii=False, indent=1)
print("\nModèle sauvegardé : /kaggle/working/sentinelle_v1.ubj et sentinelle_v1.json (onglet Output).")